In [1]:
import pandas as pd
import surprise as sp

import warnings
warnings.filterwarnings('ignore')

#### Loading movielens ratings dataset with surprise

In [2]:
reader = sp.Reader(line_format="user item rating timestamp", sep="::")

In [7]:
ratings_path = "./Raw_Data/ratings.dat"
ratings = sp.Dataset.load_from_file(ratings_path, reader=reader)


### 1. Dataset selection

#### Test of computation time with a basic KNN basic algorithm

In [10]:
from surprise.model_selection import train_test_split
trainset, testset = train_test_split(ratings, test_size=0.25)

In [11]:
algo = sp.KNNBasic()

In [12]:
algo.fit(trainset)

Computing the msd similarity matrix...
Done computing similarity matrix.


In [13]:
predictions = algo.test(testset)

In [14]:
sp.accuracy.rmse(predictions)

RMSE: 0.9261


np.float64(0.9261462301134231)

In [15]:
sp.accuracy.mae(predictions)

MAE:  0.7307


np.float64(0.7306656111114277)

#### Reduction of the original dataset for reasonable computation time

In [19]:
df_ratings = pd.read_csv(ratings_path, sep="::", names=["user","item", "rating", "timestamp"], nrows=100000)

In [20]:
df_ratings.head()

,user,item,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [23]:
df_ratings = df_ratings[["user", "item", "rating"]]

In [24]:
reader = sp.Reader(rating_scale=(1, 5))

In [25]:
ratings = sp.Dataset.load_from_df(df_ratings[["user", "item", "rating"]], reader)

#### Testing computation time with reduced dataset

In [26]:
trainset, testset = train_test_split(ratings, test_size=0.25)

In [27]:
algo.fit(trainset)

Computing the msd similarity matrix...
Done computing similarity matrix.


In [29]:
predictions = algo.test(testset)

In [30]:
sp.accuracy.rmse(predictions)

RMSE: 0.9675


np.float64(0.9675205347904009)

In [31]:
sp.accuracy.mae(predictions)

MAE:  0.7633


np.float64(0.7632785829768263)

In [34]:
# computation time with cross_validate
from surprise.model_selection import cross_validate
cross_validate(sp.KNNBasic(), ratings, measures=["RMSE", "MAE"], cv=5, verbose=True)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBasic on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9692  0.9629  0.9726  0.9717  0.9714  0.9696  0.0035  
MAE (testset)     0.7673  0.7579  0.7677  0.7693  0.7694  0.7663  0.0043  
Fit time          0.30    0.29    0.38    0.29    0.38    0.33    0.04    
Test time         2.12    1.95    1.94    1.93    1.99    1.99    0.07    


{'test_rmse': array([0.96920761, 0.96292994, 0.97255383, 0.97174334, 0.97142138]),
 'test_mae': array([0.76733643, 0.75789465, 0.7677013 , 0.7692704 , 0.76939031]),
 'fit_time': (0.2984907627105713,
  0.29410481452941895,
  0.3791956901550293,
  0.29319214820861816,
  0.3762638568878174),
 'test_time': (2.1185054779052734,
  1.9502966403961182,
  1.9435169696807861,
  1.9306809902191162,
  1.9936935901641846)}

### 2. Comparison of various algorithm performances